# Ориентация текстовых кропов: 0 или 180

Задача — для каждого изображения предсказать p 180: вероятность того, что текст перевёрнут на 180. Метрика Brier Score, важны откалиброванные вероятности.

Каскадный вариант получил 0.98809675 на закрытом тесте Avito.

## Идея решения

1. Модели ориентации оценивают исходный кроп и повернутую на 180 копию.
2. OCR модели сравнивают в какой ориентации текст читается увереннее. Распознанная строка не главная метрика, используется только локальная уверенность OCR.
3. Сигналы объединяются в базовый предикт.
4. Для уверенных прогнозов выше 0.85 сохраняется.
5. Только сложные кроп дополнительно проверяются английском OCR, латинском OCR и штуке PP-OCRv6-small.

Поворот на 180 меняет классы местами. Благодоря этому пара прогнозов объединяется в логит пространстве так, чтобы выполнялось p(R180(x)) = 1 - p(x). Веса и порог зафиксированы в config.json.

## Структура кода

Основной код находится в run. Ключевые функции: orientation_logits, ocr_margin, predict_e2, run_cascade check_submission.

In [ ]:
from pathlib import Path
from run import make_submission

## Параметры

Папка с 20 000 изображений и путь к официальному sample_submission.csv. Режим exact воспроизводит отправленный результат, sparse работает быстрее, но из-за batch-зависимости PaddleOCR может немного отличаться численно.

In [ ]:
IMAGES = Path('../images')
SAMPLE = Path('../sample_submission.csv')
OUTPUT = Path('reproduced_submission.csv')
DEVICE = 'gpu:0'
BATCH_SIZE = 128

## воспроизведение

Модели скачиваются автоматически. На моей рабочей RTX 5070 полный прогон занимает примерно 37 минут. Функция проверит формат, ID, порядок строк, NaN и диапазон вероятностей перед сохранением CSV.

In [ ]:
submission = make_submission(
    images=IMAGES,
    sample=SAMPLE,
    output=OUTPUT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    mode='exact',
)
submission.head()

## Итог

Файл reproduced_submission.csv содержит две колонки: image_id и p_180. На официальном test каскад направляет в сложную ветку 942 из 20 000 изображений. Результат полного контрольного запуска не имел ни одного hard-label отличия от отправленного CSV; максимальная разница вероятностей составила 9.11e-7.

Внешние inference API, ручная разметка test и скрытые лейблы не использовались. Все модели запускаются локально.